# Classification of a wider monoterpene synthase dataset

This notebook tests whether an XGBoost model trained on the full linear and cyclic monoterpene synthase dataset can classify enzymes producing a wider range of monoterpenes. The training data are used only to fit the model, and the wider monoterpene synthase dataset is used only for testing.

The analysis reports overall and balanced accuracy, Matthews correlation coefficient, and per-class performance. It also compares the full wider dataset with subsets containing or excluding bicyclic-product synthases.

Run this notebook from its repository folder. `FullDataset.csv` must be present in the repository, and `WiderMTtestset.csv` must be present in this folder or one of its parent folders. The wider dataset includes a `Cyclisation` column identifying each product as `Linear`, `Monocyclic`, or `Bicyclic`.


## Load packages and define the model


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from xgboost import XGBClassifier


TRAINING_FILENAME = "FullDataset.csv"
WIDER_MTS_FILENAME = "WiderMTtestset.csv"
OUTPUT_DIRECTORY = Path("wider_monoterpene_results")
FIGURE_DPI = 800
EXPECTED_BICYCLIC_COUNT = 9

SELECTED_FEATURES = [
    "Glycine - G",
    "Alanine - A",
    "Leucine - L",
    "Lysine - K",
    "Serine - S",
    "Isoleucine - I",
    "Cysteine - C",
    "Tyrosine - Y",
    "Histidine - H",
    "Aspartic Acid - D",
    "Charged",
    "Uncharged",
    "Small",
    "Aromatic",
]

XGBOOST_PARAMETERS = {
    "eval_metric": "logloss",
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbosity": 0,
    "random_state": 42,
    "tree_method": "hist",
    "learning_rate": 0.1,
}

CLASS_NAMES = ["Linear", "Cyclic"]


## Load the training and wider monoterpene datasets


In [ ]:
def find_dataset(filename, required_columns=None):
    required_columns = set(required_columns or [])
    candidates = []

    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / filename
        if candidate.is_file():
            candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            f"{filename} was not found in the current folder or its parent folders."
        )

    if not required_columns:
        return candidates[0]

    compatible_candidates = []
    for candidate in candidates:
        available_columns = set(pd.read_csv(candidate, nrows=0).columns)
        if required_columns.issubset(available_columns):
            compatible_candidates.append(candidate)

    if len(compatible_candidates) == 1:
        return compatible_candidates[0]

    if not compatible_candidates:
        locations = "\n".join(f"- {candidate}" for candidate in candidates)
        raise ValueError(
            f"Only an older copy of {filename} was found. The required columns "
            f"{sorted(required_columns)} are missing. Replace it with the updated dataset.\n"
            f"Files checked:\n{locations}"
        )

    locations = "\n".join(f"- {candidate}" for candidate in compatible_candidates)
    raise ValueError(
        f"More than one compatible copy of {filename} was found. Keep one copy so the "
        f"analysis cannot select the wrong dataset.\nFiles found:\n{locations}"
    )


training_path = find_dataset(TRAINING_FILENAME)
wider_mts_path = find_dataset(
    WIDER_MTS_FILENAME,
    required_columns=["Protein", "Top Product", "Cyclisation", "Cyclical"],
)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

training_data = pd.read_csv(training_path)
wider_mts_data = pd.read_csv(wider_mts_path)

for dataset_name, dataset in [
    ("training dataset", training_data),
    ("wider monoterpene dataset", wider_mts_data),
]:
    required_columns = [*SELECTED_FEATURES, "Cyclical"]
    if dataset_name == "wider monoterpene dataset":
        required_columns.extend(["Protein", "Top Product", "Cyclisation"])

    missing_columns = [column for column in required_columns if column not in dataset]
    if missing_columns:
        raise KeyError(f"Missing columns in the {dataset_name}: {missing_columns}")

normalized_cyclisation = (
    wider_mts_data["Cyclisation"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)
category_names = {
    "linear": "Linear",
    "monocyclic": "Monocyclic",
    "bicyclic": "Bicyclic",
}
observed_categories = set(normalized_cyclisation)
expected_categories = set(category_names)
if observed_categories - expected_categories:
    raise ValueError(
        "Unexpected cyclisation categories: "
        f"{sorted(observed_categories - expected_categories)}"
    )
if (normalized_cyclisation == "").any():
    raise ValueError("The wider monoterpene dataset contains missing cyclisation categories.")

wider_mts_data["Cyclisation"] = normalized_cyclisation.map(category_names)
bicyclic_mask = normalized_cyclisation.eq("bicyclic")
if int(bicyclic_mask.sum()) != EXPECTED_BICYCLIC_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_BICYCLIC_COUNT} bicyclic enzymes, but found "
        f"{int(bicyclic_mask.sum())} in {wider_mts_path}. "
        f"Observed categories: {wider_mts_data['Cyclisation'].value_counts().to_dict()}"
    )

X_train = training_data[SELECTED_FEATURES]
y_train = training_data["Cyclical"]
X_test = wider_mts_data[SELECTED_FEATURES]
y_test = wider_mts_data["Cyclical"]

print(f"Training dataset: {training_path} ({len(training_data)} enzymes)")
print(f"Wider monoterpene dataset: {wider_mts_path} ({len(wider_mts_data)} enzymes)")
print("Product categories:")
print(wider_mts_data["Cyclisation"].value_counts().to_string())
print("Bicyclic enzymes:")
print(wider_mts_data.loc[bicyclic_mask, ["Protein", "Top Product"]].to_string(index=False))


## Train the model on the full monoterpene dataset

Each training sample is weighted by the inverse frequency of its class. The wider monoterpene dataset is not included during model fitting.


In [ ]:
class_frequencies = y_train.value_counts(normalize=True)
sample_weights = y_train.map((1.0 / class_frequencies).to_dict())

model = XGBClassifier(**XGBOOST_PARAMETERS)
model.fit(X_train, y_train, sample_weight=sample_weights, verbose=False)

y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)

print("Model fitted on the full training dataset.")


## Plot model feature importance

Feature importance is calculated as the mean gain from splits using each selected descriptor in the model trained on the full dataset.


In [ ]:
def format_one_significant_figure(value, position):
    if value == 0:
        return "0"
    return f"{float(f'{value:.1g}'):g}"


gain_scores = model.get_booster().get_score(importance_type="gain")
if gain_scores and all(
    key.startswith("f") and key[1:].isdigit() for key in gain_scores
):
    gain_scores = {
        SELECTED_FEATURES[int(key[1:])]: value
        for key, value in gain_scores.items()
    }

feature_importance = pd.Series(
    {feature: gain_scores.get(feature, 0.0) for feature in SELECTED_FEATURES}
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(feature_importance.index, feature_importance.values, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Feature importance (gain)", fontweight="bold")
ax.invert_yaxis()
ax.xaxis.set_major_formatter(FuncFormatter(format_one_significant_figure))
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.tight_layout()
figure_path = OUTPUT_DIRECTORY / "wider_monoterpene_feature_importance_gain.png"
fig.savefig(figure_path, dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.1)
plt.show()

print(f"Saved {figure_path}")


## Evaluate predictions on the wider monoterpene dataset

Class 0 represents linear products, and class 1 represents cyclic products. Balanced accuracy is reported only when both classes are present in the evaluated subset.


In [ ]:
def evaluate_predictions(true_labels, predicted_labels, description):
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)
    matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        labels=[0, 1],
        zero_division=0,
    )

    print(f"\n{description} (n = {len(true_labels)})")
    print(f"Accuracy: {accuracy_score(true_labels, predicted_labels):.3f}")

    if np.unique(true_labels).size == 2:
        balanced_accuracy = balanced_accuracy_score(true_labels, predicted_labels)
        print(f"Balanced accuracy: {balanced_accuracy:.3f}")
        print(f"MCC: {matthews_corrcoef(true_labels, predicted_labels):.3f}")
    else:
        print("Balanced accuracy: not defined for a single-class subset")
        print("MCC: not defined for a single-class subset")

    for index, class_name in enumerate(CLASS_NAMES):
        if support[index] == 0:
            continue

        print(
            f"{class_name}: precision = {precision[index]:.3f}, "
            f"recall = {recall[index]:.3f}, "
            f"F1 = {f1[index]:.3f}, n = {support[index]}"
        )

    print(f"Confusion matrix (rows = actual, columns = predicted):\n{matrix}")
    return matrix


def plot_confusion_matrix(matrix, filename):
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.heatmap(
        matrix,
        annot=True,
        annot_kws={"fontweight": "bold"},
        cmap="Blues",
        fmt="g",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
    )
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual", fontweight="bold")

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

    fig.tight_layout()
    figure_path = OUTPUT_DIRECTORY / filename
    fig.savefig(figure_path, dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.1)
    plt.show()
    print(f"Saved {figure_path}")


full_matrix = evaluate_predictions(
    y_test,
    y_pred,
    "Wider monoterpene synthase dataset",
)
plot_confusion_matrix(full_matrix, "wider_monoterpene_confusion_matrix.png")


## Inspect individual enzyme predictions


In [ ]:
prediction_table = pd.DataFrame(
    {
        "Protein": wider_mts_data["Protein"].to_numpy(),
        "Actual class": y_test.to_numpy(),
        "Predicted class": y_pred,
        "Confidence": y_probability.max(axis=1),
        "Product": wider_mts_data["Top Product"].fillna("N/A").to_numpy(),
        "Cyclisation": wider_mts_data["Cyclisation"].to_numpy(),
    }
)

incorrect_predictions = prediction_table[
    prediction_table["Actual class"] != prediction_table["Predicted class"]
]
correct_predictions = prediction_table[
    prediction_table["Actual class"] == prediction_table["Predicted class"]
]

print("\nMisclassified enzymes:")
print(
    incorrect_predictions.to_string(index=False)
    if not incorrect_predictions.empty
    else "None"
)

print("\nCorrectly classified enzymes:")
print(
    correct_predictions.to_string(index=False)
    if not correct_predictions.empty
    else "None"
)


## Assess bicyclic-product synthases separately

Enzymes labelled `Bicyclic` in the `Cyclisation` column are evaluated separately from enzymes labelled `Linear` or `Monocyclic`.


In [ ]:
subsets = [
    (
        bicyclic_mask.to_numpy(),
        "Bicyclic-product monoterpene synthases",
        "wider_monoterpene_bicyclic_confusion_matrix.png",
    ),
    (
        (~bicyclic_mask).to_numpy(),
        "Wider monoterpene dataset excluding bicyclic-product synthases",
        "wider_monoterpene_excluding_bicyclic_confusion_matrix.png",
    ),
]

for subset_mask, description, filename in subsets:
    if not subset_mask.any():
        print(f"\nNo enzymes found for: {description}")
        continue

    subset_matrix = evaluate_predictions(
        y_test.to_numpy()[subset_mask],
        y_pred[subset_mask],
        description,
    )
    plot_confusion_matrix(subset_matrix, filename)
